# Fund Flow Inefficiency Model

goal: identify dept-quarter allocations where spending was inefficient —  
e.g. under-utilisation, q4 dumping, heavy budget revisions.

unit of analysis is one row in `budget_allocations.csv` (dept × year × quarter).  
we join with transactions to get actual spend behaviour as extra features.

In [1]:
import csv
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from collections import defaultdict
from datetime import datetime
from pathlib import Path

DATA = Path('..') / 'data'

def lc(fname):
    with open(DATA / fname, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

allocs = lc('budget_allocations.csv')
txns   = lc('transactions.csv')
ddos   = lc('ddo_accounts.csv')

print('allocations :', len(allocs))
print('transactions:', len(txns))
print('allocation cols:', list(allocs[0].keys()))

allocations : 600
transactions: 43004
allocation cols: ['id', 'dept_id', 'fiscal_year', 'quarter', 'allocated_amount', 'revised_amount', 'target_utilization_pct']


In [2]:
# sanity check on a few rows
for r in allocs[:3]:
    print(r)

{'id': '1', 'dept_id': '1', 'fiscal_year': '2022', 'quarter': '1', 'allocated_amount': '47706000', 'revised_amount': '45386000', 'target_utilization_pct': '83.4'}
{'id': '2', 'dept_id': '1', 'fiscal_year': '2022', 'quarter': '2', 'allocated_amount': '63608000', 'revised_amount': '61973000', 'target_utilization_pct': '90.2'}
{'id': '3', 'dept_id': '1', 'fiscal_year': '2022', 'quarter': '3', 'allocated_amount': '79510000', 'revised_amount': '79337000', 'target_utilization_pct': '82.8'}


In [3]:
# get spend ratios — what does the distribution look like before we pick a label threshold
spend_ratios = []
rev_ratios   = []
for r in allocs:
    alloc   = float(r['allocated_amount'])
    revised = float(r['revised_amount'])
    if alloc > 0:
        rev_ratios.append(revised / alloc)
    # no actual_spend in the CSV — we'll derive it from transactions

rev_ratios = np.array(rev_ratios)
print(f'revision ratio  — min:{rev_ratios.min():.2f}  mean:{rev_ratios.mean():.2f}  max:{rev_ratios.max():.2f}')
print(f'target_util col — sample: {[r["target_utilization_pct"] for r in allocs[:5]]}')

revision ratio  — min:0.88  mean:1.00  max:1.12
target_util col — sample: ['83.4', '90.2', '82.8', '83.3', '91.4']


In [4]:
# build DDO → dept mapping  (join key is ddo_code, not id)
ddo_dept = {d['ddo_code']: d['dept_id'] for d in ddos}

# aggregate transactions by dept × year
# txns don't have fiscal_year directly — derive from release_date
dept_year_txns      = defaultdict(list)   # (dept, year) -> [amounts]
dept_year_vendors   = defaultdict(set)    # (dept, year) -> set of vendor_ids
dept_year_q4_amt    = defaultdict(float)

for t in txns:
    dco  = t['ddo_code']
    dept = ddo_dept.get(dco, None)
    if dept is None:
        continue
    try:
        from datetime import date
        rd    = date.fromisoformat(t['release_date'])
        # fiscal year: April-March, so if month < 4 it belongs to prev year
        fy    = rd.year if rd.month >= 4 else rd.year - 1
        amt   = float(t['amount'])
        dept_year_txns[(dept, fy)].append(amt)
        dept_year_vendors[(dept, fy)].add(t['vendor_id'])
        if rd.month >= 10 or rd.month <= 3:
            dept_year_q4_amt[(dept, fy)] += amt
    except:
        pass

print(f'unique (dept, year) combos with txns: {len(dept_year_txns)}')

unique (dept, year) combos with txns: 200


In [5]:
# per-dept DDO counts
dept_ddo_count = defaultdict(int)
for d in ddos:
    dept_ddo_count[d['dept_id']] += 1

MAX_DDO_COUNT = max(dept_ddo_count.values()) or 1

# normalisation bounds we'll compute over the full alloc data
all_tx_counts   = [len(dept_year_txns.get((r['dept_id'], int(r['fiscal_year'])), []))
                   for r in allocs]
all_vendor_divs = [len(dept_year_vendors.get((r['dept_id'], int(r['fiscal_year'])), set()))
                   for r in allocs]

MAX_TX  = max(all_tx_counts)  or 1
MAX_VD  = max(all_vendor_divs) or 1

all_avg_tx = []
for r in allocs:
    amts = dept_year_txns.get((r['dept_id'], int(r['fiscal_year'])), [])
    all_avg_tx.append(np.mean(amts) if amts else 0)
MAX_AVG_TX = max(all_avg_tx) or 1

print('MAX_TX_COUNT:', MAX_TX, '  MAX_VENDOR_DIV:', MAX_VD)

MAX_TX_COUNT: 455   MAX_VENDOR_DIV: 87


## Feature engineering + labelling

inefficiency label: actual spend ratio < 0.75 AND (revised down heavily OR missed target)

In [6]:
np.random.seed(42)
tf.random.set_seed(42)

X, y = [], []

for r in allocs:
    dept    = r['dept_id']
    fy      = int(r['fiscal_year'])
    quarter = int(r['quarter'])
    alloc   = float(r['allocated_amount'])
    revised = float(r['revised_amount'])
    tgt_pct = float(r['target_utilization_pct'])

    if alloc <= 0 or revised <= 0:
        continue

    revision_ratio = revised / alloc

    # estimate actual spend from transactions (dept+year bucket)
    tx_amts = dept_year_txns.get((dept, fy), [])
    # weight spend towards this quarter (rough: quarter spend ~ total/4 scaled by tgt)
    total_tx_amt = sum(tx_amts)
    actual_spend = total_tx_amt / 4.0  # distribute evenly across quarters as proxy
    actual_spend_ratio = actual_spend / revised if revised > 0 else 0
    actual_spend_ratio = min(actual_spend_ratio, 2.0)

    target_gap = actual_spend_ratio - tgt_pct / 100.0

    # scheme type: treat depts with id ending in even number as 'capital'
    try:
        dept_num = int(str(dept).replace('DEPT', ''))
        is_capital = 1.0 if dept_num % 2 == 0 else 0.0
    except:
        is_capital = 0.0

    is_q4 = 1.0 if quarter == 4 else 0.0

    dept_tx_count_norm = len(tx_amts) / MAX_TX
    vendor_div_norm    = len(dept_year_vendors.get((dept, fy), set())) / MAX_VD
    avg_tx_size_norm   = (np.mean(tx_amts) if tx_amts else 0) / MAX_AVG_TX

    total_for_fy  = sum(dept_year_txns.get((dept, fy), []))
    q4_amt        = dept_year_q4_amt.get((dept, fy), 0)
    q4_frac       = q4_amt / total_for_fy if total_for_fy > 0 else 0.25

    # historical avg — other years for same dept
    other_year_amts = [
        np.mean(dept_year_txns[(dept, yr)])
        for yr in set(yr_ for d_, yr_ in dept_year_txns.keys() if d_ == dept)
        if yr != fy and dept_year_txns[(dept, yr)]
    ]
    hist_avg = float(np.mean(other_year_amts)) if other_year_amts else float(np.mean(tx_amts) if tx_amts else 0)
    hist_avg_norm = hist_avg / MAX_AVG_TX

    ddo_cnt_norm = dept_ddo_count.get(dept, 0) / MAX_DDO_COUNT

    features = [
        revision_ratio,
        actual_spend_ratio / 2.0,
        (target_gap + 1.0) / 2.0,
        is_capital,
        is_q4,
        dept_tx_count_norm,
        vendor_div_norm,
        avg_tx_size_norm,
        q4_frac,
        hist_avg_norm,
        ddo_cnt_norm,
    ]

    # label
    inefficient = (
        actual_spend_ratio < 0.75 and
        (revision_ratio < 0.93 or target_gap < -0.12)
    )

    X.append(features)
    y.append(1 if inefficient else 0)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

print(f'total rows  : {len(y)}')
print(f'inefficient : {int(y.sum())} ({y.mean()*100:.1f}%)')

total rows  : 600
inefficient : 178 (29.7%)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test  = sc.transform(X_test)

n_neg = (y_train == 0).sum(); n_pos = (y_train == 1).sum()
w = n_neg / max(n_pos, 1)
print(f'class weight 1:{w:.2f}   neg={n_neg}  pos={n_pos}')

class weight 1:2.38   neg=338  pos=142


In [8]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
], name='fund_flow_inefficiency')

model.compile(
    optimizer=keras.optimizers.Adam(3e-4),
    loss='binary_crossentropy',
    metrics=[
        keras.metrics.AUC(name='auc'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
    ]
)

model.summary()

Model: "fund_flow_inefficiency"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,649 (14.25 KB)

 Trainable params: 3,521 (13.75 KB)

 Non-trainable params: 128 (512.00 B)

In [9]:
cbs = [
    keras.callbacks.EarlyStopping(monitor='val_auc', patience=20,
                                  restore_best_weights=True, mode='max'),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                      patience=8, min_lr=1e-6),
]

history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=200,
    batch_size=32,
    class_weight={0: 1.0, 1: w},
    callbacks=cbs,
    verbose=1
)

Epoch 1/200


 1/13 ━━━━━━━━━━━━━━━━━━━━ 18s 2s/step - auc: 0.3889 - loss: 1.1639 - precision: 0.2800 - recall: 0.7778

13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - auc: 0.4009 - loss: 1.1276 - precision: 0.2569 - recall: 0.7241 - val_auc: 0.4753 - val_loss: 0.7090 - val_precision: 0.3594 - val_recall: 0.8846 - learning_rate: 3.0000e-04


Epoch 2/200


 1/13 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - auc: 0.5652 - loss: 1.0109 - precision: 0.3200 - recall: 0.8889

...

 1/13 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - auc: 1.0000 - loss: 0.1410 - precision: 0.9000 - recall: 1.0000

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - auc: 0.9803 - loss: 0.2627 - precision: 0.8346 - recall: 0.9138 - val_auc: 0.9778 - val_loss: 0.1908 - val_precision: 0.8333 - val_recall: 0.9615 - learning_rate: 3.0000e-04


Epoch 40/200


 1/13 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - auc: 0.9855 - loss: 0.2277 - precision: 0.8000 - recall: 0.8889

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - auc: 0.9802 - loss: 0.2607 - precision: 0.8258 - recall: 0.9397 - val_auc: 0.9774 - val_loss: 0.1904 - val_precision: 0.8333 - val_recall: 0.9615 - learning_rate: 3.0000e-04


In [10]:
y_prob = model.predict(X_test, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)
auc  = roc_auc_score(y_test, y_prob) if len(set(y_test.tolist())) > 1 else 0.0
cm   = confusion_matrix(y_test, y_pred)

print(f'Accuracy  {acc:.4f}')
print(f'Precision {prec:.4f}')
print(f'Recall    {rec:.4f}')
print(f'F1        {f1:.4f}')
print(f'AUC-ROC   {auc:.4f}')
print(f'CM:\n{cm}')

Accuracy  0.9167
Precision 0.7955
Recall    0.9722
F1        0.8750
AUC-ROC   0.9732
CM:
[[75  9]
 [ 1 35]]


In [11]:
model.save('fund_flow_inefficiency.h5')

stats = {
    'model': 'fund_flow_inefficiency',
    'framework': f'TensorFlow {tf.__version__}',
    'trained_at': datetime.utcnow().isoformat() + 'Z',
    'unit': 'department x fiscal_year x quarter',
    'samples': int(len(y)),
    'inefficiency_rate_pct': round(float(y.mean()) * 100, 2),
    'features': [
        'revision_ratio', 'actual_spend_ratio', 'target_gap',
        'is_capital', 'is_q4', 'dept_tx_count_norm',
        'vendor_diversity_norm', 'avg_tx_size_norm', 'q4_spend_frac',
        'hist_util_avg_norm', 'ddo_cnt_norm'
    ],
    'label_def': 'actual_spend_ratio < 0.75 AND (revision_ratio < 0.93 OR target_gap < -0.12)',
    'scaler_mean':  sc.mean_.tolist(),
    'scaler_scale': sc.scale_.tolist(),
    'performance': {
        'accuracy':  round(acc,  4),
        'precision': round(prec, 4),
        'recall':    round(rec,  4),
        'f1':        round(f1,   4),
        'auc_roc':   round(auc,  4),
    },
    'confusion_matrix': cm.tolist()
}

with open('fund_flow_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print('saved  fund_flow_inefficiency.h5  +  fund_flow_stats.json')

saved  fund_flow_inefficiency.h5  +  fund_flow_stats.json
